# 第77章 数据预处理与Pipeline

使用 ColumnTransformer 和 Pipeline 对数值、类别与缺失值进行一致预处理，避免训练测试之间的数据泄漏。


## 先解决一个小问题

围绕“数据预处理与Pipeline”完成一个可验证的小型建模实验：先明确输入和目标，再比较方法带来的变化。使用 ColumnTransformer 和 Pipeline 对数值、类别与缺失值进行一致预处理，避免训练测试之间的数据泄漏。


## 这章为什么先学

这是“机器学习”建模主线中的第 77 章，重点放在“数据预处理与Pipeline”对应的一个具体决策，而不是重复完整流程。


## 开始前确认

- 能够使用 pandas 读取、筛选和汇总数据
- 理解训练集、测试集和基本统计指标
- 本章会进一步练习：识别数值和类别特征、分别配置缺失填补、缩放和独热编码、用 ColumnTransformer 合并预处理


## 做完要留下什么

完成一份围绕“数据预处理与Pipeline”的可运行实验：包含数据准备、方法执行、指标或图表证据，以及一句有边界的结论。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 识别数值和类别特征
- 分别配置缺失填补、缩放和独热编码
- 用 ColumnTransformer 合并预处理
- 把预处理与模型封装为 Pipeline


## 核心概念

- 数值缩放对距离和间隔模型很重要
- OneHotEncoder 将无序类别转成指示变量
- handle_unknown 避免测试集新类别报错
- Pipeline 保证交叉验证时每折独立拟合预处理


## 示例 1：构造混合类型数据

使用 Titanic 公开数据展示真实缺失值和类别字段。


In [ ]:
import pandas as pd

url = "/datasets/titanic.csv"
df = pd.read_csv(url)
features = ['pclass', 'sex', 'age', 'fare', 'embarked']
X, y = df[features], df['survived']
print(X.dtypes)
print('缺失值:', X.isna().sum().to_dict())


## 示例 2：列级预处理流水线

所有填补和编码都封装在流水线中，只在训练集拟合。


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num = ['age', 'fare']; cat = ['pclass', 'sex', 'embarked']
prep = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)
])
pipe = Pipeline([('prep', prep), ('model', LogisticRegression(max_iter=500))])
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=77)
pipe.fit(X_train, y_train)
print('测试准确率:', round(pipe.score(X_test, y_test), 3))
print('转换后特征数:', len(pipe.named_steps['prep'].get_feature_names_out()))


## 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 常见误区

- 切分前用全量数据计算均值或标准差
- 对名义类别直接使用 1、2、3 表示大小
- 测试集出现新类别时编码器报错
- 在训练和预测阶段手工维护两套预处理代码


## 综合练习

1. 增加 sibsp 和 parch 两个数值特征
2. 重新拟合流水线
3. 输出新特征数和测试准确率

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“增加 sibsp 和 parch 两个数值特征”。
2. **独立完成**：不复制示例代码，完成“重新拟合流水线”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“输出新特征数和测试准确率”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
features2 = features + ['sibsp', 'parch']
num2 = num + ['sibsp', 'parch']
prep2 = ColumnTransformer([('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num2), ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)])
X2_train, X2_test, y2_train, y2_test = train_test_split(df[features2], y, stratify=y, random_state=77)
pipe2 = Pipeline([('prep', prep2), ('model', LogisticRegression(max_iter=500))]).fit(X2_train, y2_train)
practice_score = pipe2.score(X2_test, y2_test)
print('准确率:', round(practice_score, 3))

# 自检
assert 0 <= practice_score <= 1
assert len(pipe2.named_steps['prep'].get_feature_names_out()) >= 8


## 本章小结

使用 ColumnTransformer 和 Pipeline 对数值、类别与缺失值进行一致预处理，避免训练测试之间的数据泄漏。


### 你已经掌握

- 识别数值和类别特征
- 分别配置缺失填补、缩放和独热编码
- 用 ColumnTransformer 合并预处理
- 把预处理与模型封装为 Pipeline


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 构造混合类型数据 | 使用 Titanic 公开数据展示真实缺失值和类别字段。 | `pd.read_csv()`、`X.isna()`、`.sum()`、`.to_dict()` |
| 列级预处理流水线 | 所有填补和编码都封装在流水线中，只在训练集拟合。 | `pipe.fit()`、`pipe.score()`、`.get_feature_names_out()`、`named_steps['prep']` |


### 需要注意

- 切分前用全量数据计算均值或标准差
- 对名义类别直接使用 1、2、3 表示大小
- 测试集出现新类别时编码器报错
- 在训练和预测阶段手工维护两套预处理代码


### 完成检查

- [ ] 能够识别数值和类别特征
- [ ] 能够分别配置缺失填补、缩放和独热编码
- [ ] 能够用 ColumnTransformer 合并预处理
- [ ] 能够把预处理与模型封装为 Pipeline


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
